In [1]:
import os
os.chdir(r"E:\MyAIProject") # workspace
from LinLanAIFrame import *
from LinLanAIFrame.load_pretrain_models import get_model_path

In [2]:
def image_preprocessing(file_path: str):
    image = Image.open(file_path).convert('RGB')
    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
    ])
    return transform(image)


def text_preprocessing(text: str):
    tokenizer = Tokenizer(dim=200, numpy=True)
    token = tokenizer(text)[0]
    return torch.tensor(token)


def data_iterator(image_paths: list, texts: list):
    images = []
    tokens = []
    for ip, tp in zip(image_paths, texts):
        images.append(image_preprocessing(ip))
        tokens.append(text_preprocessing(tp))
    return torch.stack(images, dim=0), torch.stack(tokens, dim=0)
image_paths = [os.path.join("./test_data/clip/", i) for i in os.listdir("./test_data/clip/")]
texts = [i.split('/')[-1].split(".")[0] for i in image_paths]
data = data_iterator(image_paths=image_paths, texts=texts)

In [5]:
device = "cuda" if torch.cuda.is_available() else "cpu"
def init_model(model_name, download_path="./"):
    global device
    model_id, config_path, state_dict_path = get_model_path(model_name, download_path=download_path)
    model = init_model_from_config_state_dict(config_path=config_path, state_dict_path=state_dict_path, strict=False)
    for param in model.parameters():
        param.requires_grad = False
    model = model.to(device)
    model = model.eval()
    return model
clip = init_model("clip-224")
print("Using device:", device)

文件已经存在, File Exists: ./pretrain/state_dict/a542ac33c39d1e148feefdcf8f7288a7.pth
文件已经存在, File Exists: ./pretrain/config/a542ac33c39d1e148feefdcf8f7288a7.json
Using device: cuda


In [6]:
with torch.no_grad(), autocast():
    image = data[0].to(device)
    text = data[1].to(device)
    recall_result = clip.recall_at_k(image=image, text=text, top_k_arr=(1,5,10))
    for item in recall_result:
        key = list(item.keys())[0]
        display = f"{key}:image2text_acc={item[key]["image2text_acc"]:.2%}, text2image_acc={item[key]["text2image_acc"]:.2%}"
        print(display)

recall@1:image2text_acc=97.78%, text2image_acc=97.78%
recall@5:image2text_acc=97.78%, text2image_acc=100.00%
recall@10:image2text_acc=100.00%, text2image_acc=100.00%


In [7]:
quit()